# MimicVision AI v0.1 - Clase 1 | MIMIC

Pipeline completo de la primera entrega: adquisicion de datos, extraccion de landmarks,
ingenieria de caracteristicas geometricas, entrenamiento de un clasificador supervisado,
evaluacion con metricas por clase y demo en tiempo real.

El notebook corre igual en local (con el entorno de `requirements.txt`) y en Google Colab
(la primera celda detecta el entorno e instala lo necesario).

In [ ]:
# Deteccion de entorno: en Colab hay que clonar el repositorio e instalar
# dependencias; en local se asume que ya existe el entorno virtual del README.
import os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    if not Path("mimicvision-ai").exists() and not Path("mimic").exists():
        # Ajustar la URL cuando el repositorio este publicado en GitHub
        !git clone https://github.com/USUARIO/mimicvision-ai.git
    if Path("mimicvision-ai").exists():
        os.chdir("mimicvision-ai")
    !pip install -q -r requirements.txt

# Si el notebook se abrio desde la carpeta notebooks/, subimos a la raiz
# del proyecto para que los imports y las rutas relativas funcionen
if Path.cwd().name == "notebooks":
    os.chdir("..")

print("Entorno:", "Colab" if EN_COLAB else "local", "| Carpeta:", Path.cwd().name)

## 1. Datos

El dataset combina dos fuentes publicas reales (ver `docs/superpowers/specs/2026-07-05-mimic-design.md`):

- **HaGRID** (CC-BY-SA 4.0): clases `saludo` (palm) y `pulgar_arriba` (like), 120 imagenes cada una.
- **Openverse** (fotos CC0/CC-BY/CC-BY-SA): las 8 clases restantes, curadas y verificadas visualmente.

Limitacion documentada: las 8 clases curadas tienen menos de 100 muestras porque se decidio
usar unicamente fotos reales sin aumento de datos. Esto se discute en el informe final.

In [ ]:
import subprocess
import sys

# La descarga solo corre si el dataset no existe todavia: los scripts son
# reanudables y no vuelven a bajar lo que ya esta en disco.
if not Path("data/metadata.csv").exists():
    subprocess.run([sys.executable, "-m", "mimic.data_curation.descargar_hagrid"], check=True)
    subprocess.run([sys.executable, "-m", "mimic.data_curation.descargar_fotos_cc"], check=True)
    subprocess.run([sys.executable, "-m", "mimic.data_curation.construir_metadata"], check=True)

import pandas as pd

metadata = pd.read_csv("data/metadata.csv")
print(f"{len(metadata)} muestras en total")
metadata.groupby(["clase", "split"]).size().unstack(fill_value=0)

## 2. Extraccion de landmarks y features

Cada imagen pasa por MediaPipe `HolisticLandmarker` y se convierte en un vector de 10
caracteristicas geometricas normalizadas por el ancho de hombros. Se registra tambien la
tasa de deteccion por clase: las fotos donde no se detecta el torso superior quedan fuera
y esa perdida hay que conocerla, no ignorarla.

In [ ]:
import cv2
import numpy as np

from mimic.features import NARIZ, NOMBRES_FEATURES, construir_vector_features
from mimic.landmarks import DetectorHolistic

RUTA_FEATURES = Path("data/features_mimic.csv")

if RUTA_FEATURES.exists():
    features_df = pd.read_csv(RUTA_FEATURES)
else:
    detector = DetectorHolistic()
    filas = []
    fallos_por_clase = {}
    for registro in metadata.itertuples():
        imagen = cv2.imread(registro.ruta)
        if imagen is None:
            continue
        resultado = detector.procesar(imagen)
        if resultado.pose is None:
            fallos_por_clase[registro.clase] = fallos_por_clase.get(registro.clase, 0) + 1
            continue
        menton = resultado.menton if resultado.menton is not None else resultado.pose[NARIZ]
        vector = construir_vector_features(resultado.pose, menton)
        filas.append({
            "sample_id": registro.sample_id,
            "clase": registro.clase,
            "split": registro.split,
            **dict(zip(NOMBRES_FEATURES, vector)),
        })
    features_df = pd.DataFrame(filas)
    features_df.to_csv(RUTA_FEATURES, index=False)
    print("Fallos de deteccion por clase:", fallos_por_clase)

print(f"{len(features_df)} muestras con features utiles")
tasa = len(features_df) / len(metadata)
print(f"Tasa de deteccion global: {tasa:.1%}")
features_df.head()

### Diccionario de variables (`data/features_mimic.csv`)

| Variable | Descripcion |
|---|---|
| `sample_id` | Identificador de la muestra, enlaza con `data/metadata.csv` |
| `clase` | Una de las 10 categorias (5 poses + 5 gestos) |
| `split` | Particion fija: train, val o test |
| `angulo_codo_izq` / `angulo_codo_der` | Angulo del codo en grados (hombro-codo-muneca) |
| `angulo_hombro_izq` / `angulo_hombro_der` | Angulo del brazo respecto a la vertical |
| `inclinacion_cabeza` | Angulo del vector hombros-nariz respecto a la vertical |
| `distancia_manos` | Distancia entre munecas, en anchos de hombro |
| `distancia_mano_menton` | Distancia minima muneca-menton, en anchos de hombro |
| `distancia_mano_hombro_contrario` | Minimo muneca-hombro opuesto (senal de brazos cruzados) |
| `altura_manos_relativa` | Altura media de las munecas respecto a los hombros |
| `asimetria_manos` | Diferencia de altura entre munecas (gestos de una mano) |

Todas las distancias se normalizan por el ancho de hombros para que la escala de la
imagen y la distancia a la camara no afecten la clasificacion.

## 3. Entrenamiento y seleccion de modelo

Se comparan un SVM (kernel RBF) y un Random Forest mediante validacion cruzada
estratificada, y se elige por F1 macro: con clases desbalanceadas (120 muestras de HaGRID
contra 20-40 curadas), el accuracy global esconderia el mal desempeno en las clases chicas.

In [ ]:
from mimic.classifier import entrenar_y_seleccionar

entrenamiento = features_df[features_df["split"].isin(["train", "val"])]
prueba = features_df[features_df["split"] == "test"]

X_entrenamiento = entrenamiento[NOMBRES_FEATURES].to_numpy()
y_entrenamiento = entrenamiento["clase"].to_numpy()
X_prueba = prueba[NOMBRES_FEATURES].to_numpy()
y_prueba = prueba["clase"].to_numpy()

resultado = entrenar_y_seleccionar(X_entrenamiento, y_entrenamiento)
print(f"Modelo elegido: {resultado.nombre_modelo}")
print(f"F1 macro (validacion cruzada): {resultado.f1_macro_validacion:.3f}")

## 4. Evaluacion sobre el conjunto de test

Las metricas se calculan sobre muestras que el modelo nunca vio, como exige el protocolo
de integridad del proyecto.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

y_predicho = resultado.modelo.predict(X_prueba)

print(classification_report(y_prueba, y_predicho, zero_division=0))

figura, eje = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay.from_predictions(
    y_prueba, y_predicho, ax=eje, xticks_rotation=45, colorbar=False
)
eje.set_title("Matriz de confusion - conjunto de test")
plt.tight_layout()
plt.show()

## 5. Serializacion del modelo

El modelo queda en `models/mimic_clasificador.joblib` junto con un JSON de version que
registra fecha, algoritmo, metrica de validacion y version de scikit-learn (los .joblib
no son portables entre versiones distintas de sklearn, asi que hay que dejarlo escrito).

In [ ]:
import json
from datetime import date

import sklearn

from mimic.classifier import guardar_modelo

Path("models").mkdir(exist_ok=True)
guardar_modelo(resultado.modelo, Path("models/mimic_clasificador.joblib"))

version = {
    "fecha": str(date.today()),
    "algoritmo": resultado.nombre_modelo,
    "f1_macro_validacion": round(resultado.f1_macro_validacion, 4),
    "sklearn": sklearn.__version__,
    "features": NOMBRES_FEATURES,
}
Path("models/mimic_clasificador.version.json").write_text(json.dumps(version, indent=2))
print(json.dumps(version, indent=2))

## 6. Analisis de errores

Se muestran hasta 5 errores reales del conjunto de test. Para cada uno vale preguntarse:
es una pose ambigua entre dos clases parecidas (pensando vs mano en el menton), un fallo
de deteccion de landmarks, o una foto atipica dentro de su clase?

In [ ]:
errores = prueba.copy()
errores["prediccion"] = y_predicho
errores = errores[errores["clase"] != errores["prediccion"]]
print(f"{len(errores)} errores en test")

muestra_errores = errores.head(5).merge(metadata[["sample_id", "ruta"]], on="sample_id")
figura, ejes = plt.subplots(1, max(len(muestra_errores), 1), figsize=(4 * max(len(muestra_errores), 1), 5))
if len(muestra_errores) == 1:
    ejes = [ejes]
for eje, registro in zip(ejes, muestra_errores.itertuples()):
    imagen = cv2.cvtColor(cv2.imread(registro.ruta), cv2.COLOR_BGR2RGB)
    eje.imshow(imagen)
    eje.set_title(f"real: {registro.clase}\npredicho: {registro.prediccion}", fontsize=9)
    eje.axis("off")
plt.tight_layout()
plt.show()

## 7. Latencia y FPS

Se mide el pipeline completo (landmarks + features + clasificacion) sobre imagenes reales
del conjunto de test. En una laptop tipica sin GPU el sistema deberia superar los 10 FPS,
suficiente para la demo en vivo.

In [ ]:
import time

from mimic.pipeline import procesar_frame

detector_bench = DetectorHolistic()
rutas_bench = metadata[metadata["split"] == "test"]["ruta"].head(30)

latencias = []
for ruta in rutas_bench:
    imagen = cv2.imread(ruta)
    if imagen is None:
        continue
    inicio = time.perf_counter()
    procesar_frame(imagen, detector_bench, resultado.modelo, timestamp=0.0)
    latencias.append(time.perf_counter() - inicio)

latencias = np.array(latencias)
tabla_latencia = pd.DataFrame({
    "latencia media (ms)": [latencias.mean() * 1000],
    "latencia p95 (ms)": [np.percentile(latencias, 95) * 1000],
    "FPS estimados": [1.0 / latencias.mean()],
}).round(1)
tabla_latencia

## 8. Demo en tiempo real

La interfaz usa la camara del navegador, asi que funciona igual en local y en Colab.
En local tambien se puede correr `python -m mimic.live_demo` (OpenCV puro) para medir
los FPS reales sin la latencia del navegador.

In [ ]:
from app.app import construir_demo

construir_demo().launch(share=EN_COLAB)

## Continuidad hacia la Clase 2 (MATCH)

El contrato de salida que reutilizara la Clase 2 es `mimic.pipeline.procesar_frame`, que
devuelve por cada frame: la imagen, los landmarks normalizados, la etiqueta de pose con su
confianza y el timestamp. El modulo MATCH construira sobre esto la galeria visual y el
motor de recuperacion por similitud sin modificar nada de MIMIC.